# ema-first-moment composite — cx25: full Adam step: dual EMAs + bias correction + param update

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `ema-first-moment`, `ema-second-moment`, `bias-correction-divide`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-first-moment"
DD_ATOM_IDS = ["ema-first-moment", "ema-second-moment", "bias-correction-divide"]
DD_SUBTOPICS = ["Optimizer: Adam EMA first moment", "Optimizer: Adam EMA second moment", "Optimizer: Adam bias-correction divide"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these three atoms compose

Adam threads three operations through every parameter:
1. **EMA of the gradient** (`m`, first moment) — `m = beta1*m + (1-beta1)*g`. Atom: `ema-first-moment`.
2. **EMA of the squared gradient** (`v`, second moment) — `v = beta2*v + (1-beta2)*g*g`. Atom: `ema-second-moment`.
3. **Bias correction** — both EMAs start at 0, so early steps are systematically too small. Divide by `(1 - beta**t)` to undo the warm-up bias: `m_hat = m / (1 - beta1**t)`, `v_hat = v / (1 - beta2**t)`. Atom: `bias-correction-divide`.

Then the parameter update is `p -= lr * m_hat / (sqrt(v_hat) + eps)`.

**Why all three together.** Either EMA alone is just a one-sided estimator — `m_hat` gives a momentum-style direction, `v_hat` gives a per-coordinate learning-rate scale. Bias correction is what makes them comparable to the true expected gradient and squared gradient at step `t`. Skip it and the first ~1/(1-beta) steps shrink the effective lr by a factor of (1 - beta**t).

**Anatomy of one Adam step.**
```python
t_step += 1
for p, m, v in zip(params, ms, vs):
    g = p.grad
    m.mul_(beta1).add_(g, alpha=1 - beta1)            # ema-first-moment.
    v.mul_(beta2).addcmul_(g, g, value=1 - beta2)     # ema-second-moment.
    m_hat = m / (1 - beta1 ** t_step)                  # bias-correction-divide.
    v_hat = v / (1 - beta2 ** t_step)                  # bias-correction-divide.
    p.data.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
```

We will cross-check the output against `torch.optim.Adam` — the reference Adam in PyTorch.

### Composite Exercise — full Adam step: dual EMAs + bias correction + param update

**Atoms exercised together**: `ema-first-moment`, `ema-second-moment`, `bias-correction-divide`

Implement `cx25_adam_step(params, ms, vs, t_step, lr, beta1, beta2, eps)`.

Inputs:
- `params` — list of `t.Tensor` parameters, each with `.grad` already populated.
- `ms`, `vs` — lists of running buffers, same shapes as the params, holding the current `m` and `v` EMAs.
- `t_step` — the **post-increment** step counter (i.e. for the first step, pass `t_step=1`, not `0`). Used in `(1 - beta**t_step)`.
- `lr`, `beta1`, `beta2`, `eps` — Adam hyperparameters.

Required behaviour for each `(p, m, v)`:
1. Update `m` IN-PLACE with the first-moment EMA: `m = beta1*m + (1-beta1)*g`.
2. Update `v` IN-PLACE with the second-moment EMA: `v = beta2*v + (1-beta2)*g*g`.
3. Compute `m_hat = m / (1 - beta1**t_step)`, `v_hat = v / (1 - beta2**t_step)` (may be NEW tensors — they're scratch).
4. Update `p` IN-PLACE: `p -= lr * m_hat / (sqrt(v_hat) + eps)`.

The function should return `None` — it mutates `params`, `ms`, `vs` in place. Run it under `t.inference_mode()` (or `t.no_grad()`) internally so the in-place leaf update on `p` is legal.

The test takes 5 Adam steps with random gradients and cross-checks against `torch.optim.Adam` — your `params` must end up element-wise equal to the reference.

In [ ]:
def cx25_adam_step(params, ms, vs, t_step, lr, beta1, beta2, eps):
    # In-place leaf updates on a requires_grad=True tensor need inference_mode/no_grad.
    with t.inference_mode():
        for p, m, v in zip(params, ms, vs):
            g = p.grad
            # Atom A (ema-first-moment): m <- beta1*m + (1-beta1)*g.
            m.mul_(beta1).add_(g, alpha=1 - beta1)
            # Atom B (ema-second-moment): v <- beta2*v + (1-beta2)*g*g.
            v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
            # Atom C (bias-correction-divide): undo warm-up bias of both EMAs.
            bc1 = 1 - beta1 ** t_step
            bc2 = 1 - beta2 ** t_step
            m_hat = m / bc1
            v_hat = v / bc2
            # In-place param update: p -= lr * m_hat / (sqrt(v_hat) + eps).
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)


<details><summary>Show solution — cx25</summary>

```python
def cx25_adam_step(params, ms, vs, t_step, lr, beta1, beta2, eps):
    # In-place leaf updates on a requires_grad=True tensor need inference_mode/no_grad.
    with t.inference_mode():
        for p, m, v in zip(params, ms, vs):
            g = p.grad
            # Atom A (ema-first-moment): m <- beta1*m + (1-beta1)*g.
            m.mul_(beta1).add_(g, alpha=1 - beta1)
            # Atom B (ema-second-moment): v <- beta2*v + (1-beta2)*g*g.
            v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
            # Atom C (bias-correction-divide): undo warm-up bias of both EMAs.
            bc1 = 1 - beta1 ** t_step
            bc2 = 1 - beta2 ** t_step
            m_hat = m / bc1
            v_hat = v / bc2
            # In-place param update: p -= lr * m_hat / (sqrt(v_hat) + eps).
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
```

Cross-checking against `torch.optim.Adam` is the gold standard — PyTorch's reference Adam uses exactly this update (per-param `m`, `v`, bias-corrected via division). The `v_hat.sqrt().add_(eps)` is an in-place mutation of the scratch tensor (no aliasing risk because `v_hat` was created by the `/` op). If your delta on step 1 does NOT equal `-lr` for a unit gradient, you're almost certainly missing the bias correction — without it, the step-1 delta is `-lr * (1 - beta1) / sqrt(1 - beta2)` ≈ `-lr * 0.1 / 0.0316` ≈ `-3.16 * lr`. Counter-intuitively, MISSING bias correction makes the FIRST step way too big, not too small (because `sqrt(v_hat)` is divided by a smaller number than `m_hat`).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx25'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx25',
        'subtopics': ["Optimizer: Adam EMA first moment", "Optimizer: Adam EMA second moment", "Optimizer: Adam bias-correction divide"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()